# 实践 9.2  MCAR与低丰度缺失下的插补比较

《统计遗传与智能育种学》第九章 · 电子资源 `09b_missing_imputation.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：09b_missing_imputation.ipynb。实践先生成无缺失的参照丰度矩阵，再分别施加完全随机遮蔽和低丰度依赖缺失。代码比较中位数、k近邻（k-nearest neighbors, KNN）和低值随机插补的NRMSE，并把误差评价限定在具有已知真值的遮蔽位置。练习要求改变低丰度缺失概率与遮蔽比例，观察不同插补方法的相对性能是否随缺失机制改变。

运行检查：重启内核并运行全部单元  
预期：分别输出MCAR与低丰度缺失场景的缺失率、NRMSE和方法排名；解释为何一种插补方法不能脱离缺失机制被称为普遍最优。

## 输入、完整代码、实际输出与解释

输入与可复现条件：先生成完整真值丰度矩阵，再分别施加MCAR与低丰度依赖缺失；SEED=902；NRMSE只在已知真值的人工遮蔽位置计算。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=902
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 902


In [2]:
from sklearn.impute import KNNImputer
ns,nf=180,55; truth=rng.normal(5,1.2,(ns,nf))+rng.normal(0,.6,(ns,1)); scale=np.std(truth)
def mask_data(kind,rate):
 if kind=='MCAR': mask=rng.random(truth.shape)<rate
 else:
  prob=1/(1+np.exp((truth-4.5)*3)); c=rate/prob.mean(); mask=rng.random(truth.shape)<np.clip(c*prob,0,.95)
 X=truth.copy(); X[mask]=np.nan; return X,mask
def methods(X):
 med=np.where(np.isnan(X),np.nanmedian(X,axis=0),X); knn=KNNImputer(n_neighbors=5).fit_transform(X); lod=np.nanquantile(X,.1,axis=0); left=X.copy(); ii=np.where(np.isnan(left)); left[ii]=rng.normal(lod[ii[1]],.25); return {'median':med,'KNN':knn,'left_censor_random':left}
for kind in ['MCAR','low_abundance']:
 X,mask=mask_data(kind,.18); print('mechanism',kind,'masked',round(mask.mean(),3))
 for name,Z in methods(X).items(): print(name,'NRMSE',round(np.sqrt(np.mean((Z[mask]-truth[mask])**2))/scale,3))


mechanism MCAR masked 0.182
median NRMSE 1.011
KNN NRMSE 0.995
left_censor_random NRMSE 1.636
mechanism low_abundance masked 0.173
median NRMSE 1.343
KNN NRMSE 1.252
left_censor_random NRMSE 0.676


结果解释：MCAR下KNN NRMSE 0.995略优于中位数1.011，而低值随机插补1.636较差；低丰度缺失下低值随机插补0.676反而明显优于KNN 1.252和中位数1.343。该反转说明插补方法的相对性能依赖缺失机制；真实数据的缺失位置没有可直接观察的真值，因此不能据此无条件指定单一最优插补方法。

可修改练习：改变低丰度缺失概率与遮蔽比例，画出三种方法的NRMSE曲线；增加一个与批次相关的缺失机制，检查MCAR结论是否仍成立。